<a href="https://colab.research.google.com/github/Fares-pr0g/ML-journey-ep-3-Experimenting-with-Transformers-and-Intro-to-LLM-s/blob/main/Project%202%3A%20Fine%20tuning%20the%20BERT%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import shutil
import gzip

url= "https://github.com/rasbt/machine-learning-book/raw/main/ch08/movie_data.csv.gz"

filename= url.split("/")[-1]

with open(filename, "wb") as f:
  r=requests.get(url)
  f.write(r.content)

with gzip.open('movie_data.csv.gz', 'rb') as f_in:
  with open('movie_data.csv', 'wb') as f_out:
    shutil.copyfileobj(f_in, f_out)

In [2]:
# device agnostic code
import torch

device= 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
import pandas as pd

df= pd.read_csv('movie_data.csv')
df.head(3)


,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


In [4]:
df.shape

(50000, 2)

In [5]:
# Let's extract the reviews and the labels as numpy arrays & split the data
train_texts= df.iloc[:35000]['review'].values
train_labels= df.iloc[:35000]['sentiment'].values

valid_texts= df.iloc[35000:40000]['review'].values
valid_labels= df.iloc[35000:40000]['sentiment'].values

test_texts= df.iloc[40000:]['review'].values
test_labels= df.iloc[40000:]['sentiment'].values


In [6]:
# tokenization respecting the BERT's model tokenization method
from transformers import DistilBertTokenizerFast

tokenizer= DistilBertTokenizerFast.from_pretrained(
    'distilbert-base-uncased'
)

train_encodings= tokenizer(list(train_texts), truncation= True, padding= True)

valid_encodings= tokenizer(list(valid_texts), truncation= True, padding= True)

test_encodings= tokenizer(list(test_texts), truncation= True, padding= True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
vocab= tokenizer.get_vocab()
len(vocab)

30522

In [9]:
# Let's set the dataset structure and set the dataloader
from torch.utils.data import Dataset, DataLoader

class IMDbDataset(Dataset):

  def __init__(self, encodings, labels):
    self.encodings= encodings
    self.labels= labels

  def __getitem__(self, idx):
    item= {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    item['labels']= torch.tensor(self.labels[idx])
    return item

  def __len__(self):
    return len(self.labels)

# intanciate the datasets
train_dataset= IMDbDataset(train_encodings, train_labels)
valid_dataset= IMDbDataset(valid_encodings, valid_labels)
test_dataset= IMDbDataset(test_encodings, test_labels)

# instanciate the dataloaders
train_loader= DataLoader(train_dataset, batch_size= 16, shuffle= True)
valid_loader= DataLoader(valid_dataset, batch_size= 16, shuffle= False)
test_loader= DataLoader(test_dataset, batch_size= 16, shuffle= False)


In [21]:
# test sample to preview the architecture of the data

print(next(iter(train_loader))['input_ids'].shape)
print("\n\nOne Data Sample:\n\n")
train_dataset[0]

torch.Size([16, 512])


One Data Sample:




{'input_ids': tensor([  101,  1999,  3326,  1010,  1996, 10563,  9246,  9587, 20959,  1006,
          8538,  4519,  1007,  5829,  2000,  1996,  2152,  1011,  2465,  2181,
          1997,  9852,  4033,  1010, 13861,  1010,  6117,  1012,  2006,  1996,
         25166,  2305,  1010,  6574,  1997, 14414,  1010,  2016,  2001,  7129,
          1999,  1996, 16125,  1997,  2014,  2160,  1998,  2014,  4028,  2815,
          4895, 19454,  7178,  1012,  3174,  1011,  2048,  2086,  2101,  1010,
          1996,  3213,  2928, 11865,  8093,  2386,  1006,  5696, 11463, 10698,
          1007,  1010,  2040,  2003,  1037,  2280,  2474,  6317,  2008,  2038,
          5357,  1999, 29591,  2005,  2566,  9103,  2854,  1999,  1051,  1012,
          1046,  1012,  9304,  3979,  1998,  2333,  2000,  9795,  1010,  7288,
          2000,  8556,  1996,  2553,  2007,  2010,  4256,  4459,  3134,  1006,
          4080,  6395,  1007,  2007,  1996,  3800,  1997,  3015,  1037,  2338,
          1012,  1996, 10575,  5490, 10

## Loading and fine-tuning a pre-trained BERT model

In [12]:
from transformers import DistilBertForSequenceClassification
model= DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased'
)
model.to(device)  # device agnostic code



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [22]:
# Let's set the right environement for training

# Let's first set the accuracy function

def accuracy_fn(model, loader,device):

  with torch.inference_mode():
    correct_pred, num_examples= 0,0

    for idx, batch in enumerate(loader):
      input_ids= batch['input_ids'].to(device)
      attention_mask= batch['attention_mask'].to(device)
      labels= batch['labels'].to(device)

      outputs= model(input_ids, attention_mask= attention_mask)
      logits= outputs['logits']

      predicted_labels= torch.argmax(logits, dim=1)
      num_examples+= labels.size(0)
      correct_pred+= (predicted_labels == labels).sum()

    return correct_pred.float()/num_examples*100

In [25]:
# Let's get to the fine tuning (aka the training loop):

from timeit import default_timer as timer

optim= torch.optim.Adam(model.parameters(), lr=5e-5)
NUM_EPOCHS=3
start_time= timer()

for epoch in range(NUM_EPOCHS):

  model.train()
  for idx, batch in enumerate(train_loader):

    # Prepare data
    input_ids= batch['input_ids'].to(device)
    attention_mask=batch['attention_mask'].to(device)
    labels= batch['labels'].to(device)

    # forward pass
    outputs= model(input_ids, attention_mask= attention_mask, labels= labels)
    loss, logits= outputs['loss'], outputs['logits']

    #backpropagation
    optim.zero_grad()
    loss.backward()
    optim.step()

    #Logging  ---- BTW there is 2188 batchs in total in the train_loader
    if idx % 250 ==0:
      print(f'Epoch: {epoch+1}/{NUM_EPOCHS} | Batch: {idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

  model.eval()
  with torch.inference_mode():
    train_acc= accuracy_fn(model, train_loader, device= device)
    print(f'Training accuracy: {train_acc:.2f}%')
    valid_acc= accuracy_fn(model, valid_loader, device= device)
    print(f'Validation accuracy: {valid_acc:.2f}%')

  # Log time
  print(f'Time elapsed: {(timer() - start_time)/60:.3f} min\n')

print(f'Total training time: {(timer() - start_time)/60:.3f} min')

print(f'Test Accuracy: {accuracy_fn(model, test_loader, device= device):.2f}%')


Epoch: 1/3 | Batch: 0/2188 | Loss: 0.6989
Epoch: 1/3 | Batch: 250/2188 | Loss: 0.5207
Epoch: 1/3 | Batch: 500/2188 | Loss: 0.2290
Epoch: 1/3 | Batch: 750/2188 | Loss: 0.1661
Epoch: 1/3 | Batch: 1000/2188 | Loss: 0.1478
Epoch: 1/3 | Batch: 1250/2188 | Loss: 0.1548
Epoch: 1/3 | Batch: 1500/2188 | Loss: 0.1186
Epoch: 1/3 | Batch: 1750/2188 | Loss: 0.3175
Epoch: 1/3 | Batch: 2000/2188 | Loss: 0.1314
Training accuracy: 94.95%
Validation accuracy: 90.64%
Time elapsed: 38.271 min

Epoch: 2/3 | Batch: 0/2188 | Loss: 0.2121
Epoch: 2/3 | Batch: 250/2188 | Loss: 0.0968
Epoch: 2/3 | Batch: 500/2188 | Loss: 0.0492
Epoch: 2/3 | Batch: 750/2188 | Loss: 0.1160
Epoch: 2/3 | Batch: 1000/2188 | Loss: 0.1362
Epoch: 2/3 | Batch: 1250/2188 | Loss: 0.2046
Epoch: 2/3 | Batch: 1500/2188 | Loss: 0.1868
Epoch: 2/3 | Batch: 1750/2188 | Loss: 0.0205
Epoch: 2/3 | Batch: 2000/2188 | Loss: 0.0339
Training accuracy: 98.10%
Validation accuracy: 91.60%
Time elapsed: 76.499 min

Epoch: 3/3 | Batch: 0/2188 | Loss: 0.0388


# Testing on New Data

In [30]:
# Let's create a function that takes a new review and utilizes the trained model

def test_comment(model):

  comment= input("Write a comment about a movie you have recently watched!")

  inputs= tokenizer(comment, truncation= True, padding= True, return_tensors= 'pt')

  input_ids= inputs['input_ids'].to(device)
  attention_mask= inputs['attention_mask'].to(device)
  model.eval()
  with torch.inference_mode():

    outputs= model(input_ids, attention_mask= attention_mask)
    logits= outputs['logits']
    predicted_label= torch.argmax(logits, dim=1)
    pred_label= 'positive 😊' if predicted_label.item() else 'negative 💔'

    print(f'Comment: {comment}')
    print(f'Predicted Label: {pred_label}')


In [35]:
# Test 1:
test_comment(model)

Write a comment about a movie you have recently watched!I like the movie but the plot was so expected so I would probably give it something like 7/10
Comment: I like the movie but the plot was so expected so I would probably give it something like 7/10
Predicted Label: positive 😊


In [36]:
# Test 2:
test_comment(model)

Write a comment about a movie you have recently watched!The movie was ass ngl
Comment: The movie was ass ngl
Predicted Label: negative 💔


In [37]:
# Test 3: the hardest
test_comment(model)

Write a comment about a movie you have recently watched!i don't know how I feel exactly about the movie, my friends like it so much but personally since I'm not much into k-drama, I don't think it deserves all that hype. And yeah you guessed it, I'm talking about Parasite
Comment: i don't know how I feel exactly about the movie, my friends like it so much but personally since I'm not much into k-drama, I don't think it deserves all that hype. And yeah you guessed it, I'm talking about Parasite
Predicted Label: negative 💔


## ***-> Great results!!***